In [ ]:
from pathlib import Path
import os

REPO_ROOT = Path(os.path.abspath('')).resolve()
while not (REPO_ROOT / "caseRag_engine.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

In [ ]:
! pip install datasets huggingface_hub

# CaseRag Evaluation Datasets
Benchmark exploration: IL-PCSR and FIRE 2019 AILA.

In [ ]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd

load_dotenv()
HF_TOKEN = os.environ['HF_ACCESS_TOKEN']
print('Token loaded:', HF_TOKEN[:8], '...')

---
# 1. IL-PCSR
**Indian Legal Prior Case and Statute Retrieval** (EMNLP 2025, arXiv 2511.00268)
20k Indian Kanoon docs -- 627 test queries, 3183 precedent candidates, 936 statute candidates.

In [ ]:
queries    = load_dataset('Exploration-Lab/IL-PCSR', 'queries',    token=HF_TOKEN)
statutes   = load_dataset('Exploration-Lab/IL-PCSR', 'statutes',   token=HF_TOKEN)
precedents = load_dataset('Exploration-Lab/IL-PCSR', 'precedents', token=HF_TOKEN)

print(queries)
print(statutes)
print(precedents)

In [ ]:
# Peek at one test query
sample = queries['test_queries'][0]

sample

In [ ]:
# Extract Facts + Issue segments only (what we feed into CaseRag as query)
def extract_query(sample):
    keep = {'Facts', 'Issue'}
    segments = [
        seg for seg, role in zip(sample['text'], sample['rhetorical_roles'])
        if role in keep
    ]
    return ' '.join(segments)

query_text = extract_query(sample)
print(f'Query length: {len(query_text.split())} words')
print(query_text[:500], '...')

In [ ]:
df_precedents = pd.DataFrame(precedents['precedent_candidates'])
df_precedents.head(1)

## 1.2 IL-PCSR to CaseRag Adapter
Map IL-PCSR rhetorical roles to CaseRag JSON keys.

In [ ]:
ROLE_MAP = {
    'Facts'                  : 'Fact',
    'Issue'                  : 'Issue',
    'Argument by Petitioner' : "Petitioner's Argument",
    'Argument by Respondent' : "Respondent's Argument",
    'Statute Analysis'       : 'Analysis of the law',
    'Precedent Analysis'     : 'Precedent Analysis',
    'Court Reasoning'        : "Court's Reasoning",
    'Conclusion'             : 'Conclusion',
    'Court Disclosure'       : 'Court Disclosure',
    'NONE'                   : 'Uncategorized',
}

def ilpcsr_to_caserag(record):
    doc = {
        'caseID'               : str(record['id']),
        'title'                : record.get('case_title', ''),
        'court'                : record.get('jurisdiction', ''),
        'date'                 : record.get('date', ''),
        'author'               : '',
        'bench'                : '',
        'relevant_statutes'    : record.get('relevant_statutes', []),
        'relevant_statute_ids' : record.get('relevant_statute_ids', []),
        'relevant_precedents'  : record.get('relevant_precedents', []),
        'relevant_precedent_ids': record.get('relevant_precedent_ids', []),
    }
    for key in ROLE_MAP.values():
        doc[key] = []
    for seg, role in zip(record['text'], record['rhetorical_roles']):
        mapped = ROLE_MAP.get(role)
        if mapped:
            doc[mapped].append(seg)
    return doc


In [ ]:
precedents['precedent_candidates']

In [ ]:
# Test on first precedent candidate
sample_prec = precedents['precedent_candidates'][0]
converted   = ilpcsr_to_caserag(sample_prec)

print('caseID :', converted['caseID'])
print('title  :', converted['title'])
print('court  :', converted['court'])
print()
for key in list(converted.keys())[3:]:
    segs = converted[key]
    print(f'{key:30s}: {len(segs)} segments | first 80 chars: {segs[0][:80] if segs else "(empty)"}')


In [ ]:
converted

In [ ]:
# Check if IL-PCSR IDs match Indian Kanoon IDs
# If yes, existing CaseRag index may already contain these docs
sample_ids = [str(precedents['precedent_candidates'][i]['id']) for i in range(5)]
print('Sample IL-PCSR precedent IDs:', sample_ids)
print()
print('Check these at indiankanoon.org/doc/<id>/ to confirm IK origin')

# Also check query IDs
q_ids = [str(queries['test_queries'][i]['id']) for i in range(5)]
print('Sample IL-PCSR query IDs    :', q_ids)


In [ ]:
# Unique rhetorical roles across ALL rows in queries + precedents
from itertools import chain

all_roles = set(chain.from_iterable(precedents['precedent_candidates']['rhetorical_roles']))
all_roles |= set(chain.from_iterable(queries['train_queries']['rhetorical_roles']))
all_roles |= set(chain.from_iterable(queries['test_queries']['rhetorical_roles']))

print('Unique roles:')
for r in sorted(all_roles):
    print(' ', r)


## 1.3 Export to CaseRag JSON
Convert corpus + query records to CaseRag format and save to disk.

In [ ]:
import json, os
from tqdm.auto import tqdm

OUT_DIR = str(REPO_ROOT / 'il-pcsr_dataset')
CORPUS_DIR  = os.path.join(OUT_DIR, 'corpus')
QUERIES_DIR = os.path.join(OUT_DIR, 'queries')
os.makedirs(CORPUS_DIR,  exist_ok=True)
os.makedirs(QUERIES_DIR, exist_ok=True)

# Convert + save all 3183 precedent candidates
for record in tqdm(precedents['precedent_candidates'], desc='Exporting corpus'):
    doc = ilpcsr_to_caserag(record)
    path = os.path.join(CORPUS_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(CORPUS_DIR))} corpus files to {CORPUS_DIR}')

In [ ]:
# Convert + save 627 test queries with gold labels
for record in tqdm(queries['test_queries'], desc='Exporting queries'):
    doc = ilpcsr_to_caserag(record)
    # Attach gold labels for evaluation
    doc['relevant_precedent_ids'] = record['relevant_precedent_ids']
    doc['relevant_statute_ids']   = record['relevant_statute_ids']
    path = os.path.join(QUERIES_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(QUERIES_DIR))} query files to {QUERIES_DIR}')

# Peek at one converted query
sample_path = os.path.join(QUERIES_DIR, os.listdir(QUERIES_DIR)[0])
with open(sample_path) as f:
    sample_out = json.load(f)
print(list(sample_out.keys()))
print('Gold precedents:', sample_out['relevant_precedent_ids'])


In [ ]:
import json, os
from tqdm.auto import tqdm
OUT_DIR   = str(REPO_ROOT / 'il-pcsr_dataset')
TRAIN_DIR = os.path.join(OUT_DIR, 'train_queries')
DEV_DIR   = os.path.join(OUT_DIR, 'dev_queries')
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(DEV_DIR,   exist_ok=True)

for record in tqdm(queries['train_queries'], desc='Exporting train queries'):
    doc = ilpcsr_to_caserag(record)
    doc['relevant_precedent_ids'] = record['relevant_precedent_ids']
    doc['relevant_statute_ids']   = record['relevant_statute_ids']
    path = os.path.join(TRAIN_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(TRAIN_DIR))} train query files to {TRAIN_DIR}')

for record in tqdm(queries['dev_queries'], desc='Exporting dev queries'):
    doc = ilpcsr_to_caserag(record)
    doc['relevant_precedent_ids'] = record['relevant_precedent_ids']
    doc['relevant_statute_ids']   = record['relevant_statute_ids']
    path = os.path.join(DEV_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(DEV_DIR))} dev query files to {DEV_DIR}')

---
# 2. FIRE 2019 AILA
**Forum for Information Retrieval Evaluation -- Legal Track**
Approx 3000 Supreme Court of India docs, 50 queries, binary TREC qrels. Free, no registration (Zenodo CC-BY 4.0).

In [ ]:
import requests, zipfile, io, os

FIRE_DIR = str(REPO_ROOT / 'fire_2019')
ZENODO   = 'https://zenodo.org/record/4063986/files/AILA_2019_dataset.zip?download=1'  # zenodo.org/records/4063986 -> AILA_2019_dataset.zip
os.makedirs(FIRE_DIR, exist_ok=True)

if not os.listdir(FIRE_DIR):
    print('Downloading FIRE 2019 (21MB)...')
    resp = requests.get(ZENODO, stream=True)
    zf   = zipfile.ZipFile(io.BytesIO(resp.content))
    zf.extractall(FIRE_DIR)
    print('Done.')
else:
    print('Already downloaded.')

for root, dirs, files in os.walk(FIRE_DIR):
    lvl = root.replace(FIRE_DIR, '').count(os.sep)
    print('  ' * lvl + os.path.basename(root) + '/')
    for fn in files[:3]: print('  ' * (lvl+1) + fn)
    if len(files) > 3: print('  ' * (lvl+1) + f'... ({len(files)} files total)')

# CaseRag Evaluation Datasets
Benchmark exploration: IL-PCSR and FIRE 2019 AILA.

In [ ]:
import os
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd

load_dotenv()
HF_TOKEN = os.environ['HF_ACCESS_TOKEN']
print('Token loaded:', HF_TOKEN[:8], '...')

---
# 1. IL-PCSR
**Indian Legal Prior Case and Statute Retrieval** (EMNLP 2025, arXiv 2511.00268)
20k Indian Kanoon docs -- 627 test queries, 3183 precedent candidates, 936 statute candidates.

In [ ]:
queries    = load_dataset('Exploration-Lab/IL-PCSR', 'queries',    token=HF_TOKEN)
statutes   = load_dataset('Exploration-Lab/IL-PCSR', 'statutes',   token=HF_TOKEN)
precedents = load_dataset('Exploration-Lab/IL-PCSR', 'precedents', token=HF_TOKEN)

print(queries.column_names)
print(statutes.column_names)
print(precedents.column_names)

### EDA

In [ ]:
# Peek at one test query
sample = queries['dev_queries'][0]

sample

In [ ]:
# Extract Facts + Issue segments only (what we feed into CaseRag as query)
def extract_query(sample):
    keep = {'Facts', 'Issue'}
    segments = [
        seg for seg, role in zip(sample['text'], sample['rhetorical_roles'])
        if role in keep
    ]
    return ' '.join(segments)

query_text = extract_query(sample)
print(f'Query length: {len(query_text.split())} words')
print(query_text[:500], '...')

In [ ]:
df_precedents = pd.DataFrame(precedents['precedent_candidates'])
df_precedents.head(1)

In [ ]:
queries

In [ ]:
len(statutes['statute_candidates'])

### 1.2 IL-PCSR to CaseRag Adapter
Map IL-PCSR rhetorical roles to CaseRag JSON keys.

In [ ]:
ROLE_MAP = {
    'Facts'                  : 'Fact',
    'Issue'                  : 'Issue',
    'Argument by Petitioner' : "Petitioner's Argument",
    'Argument by Respondent' : "Respondent's Argument",
    'Statute Analysis'       : 'Analysis of the law',
    'Precedent Analysis'     : 'Precedent Analysis',
    'Court Reasoning'        : "Court's Reasoning",
    'Conclusion'             : 'Conclusion',
    'Court Disclosure'       : 'Court Disclosure',
    'NONE'                   : 'Uncategorized',
}

def ilpcsr_to_caserag(record):
    doc = {
        'caseID'               : str(record['id']),
        'title'                : record.get('case_title', ''),
        'court'                : record.get('jurisdiction', ''),
        'date'                 : record.get('date', ''),
        'author'               : '',
        'bench'                : '',
        'relevant_statutes'    : record.get('relevant_statutes', []),
        'relevant_statute_ids' : record.get('relevant_statute_ids', []),
        'relevant_precedents'  : record.get('relevant_precedents', []),
        'relevant_precedent_ids': record.get('relevant_precedent_ids', []),
    }
    for key in ROLE_MAP.values():
        doc[key] = []
    for seg, role in zip(record['text'], record['rhetorical_roles']):
        mapped = ROLE_MAP.get(role)
        if mapped:
            doc[mapped].append(seg)
    return doc


In [ ]:

import json

sample_prec = precedents['precedent_candidates'][0]
converted   = ilpcsr_to_caserag(sample_prec)
print(json.dumps(converted, indent=2, ensure_ascii=False))

## 1.3 Export to CaseRag JSON
Convert corpus + query records to CaseRag format and save to disk.

In [ ]:
import json, os
from tqdm.auto import tqdm

OUT_DIR = r'il-pcsr_dataset'
CORPUS_DIR  = os.path.join(OUT_DIR, 'corpus')
QUERIES_DIR = os.path.join(OUT_DIR, 'queries')
os.makedirs(CORPUS_DIR,  exist_ok=True)
os.makedirs(QUERIES_DIR, exist_ok=True)

# Convert + save all 3183 precedent candidates
for record in tqdm(precedents['precedent_candidates'], desc='Exporting corpus'):
    doc = ilpcsr_to_caserag(record)
    path = os.path.join(CORPUS_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(CORPUS_DIR))} corpus files to {CORPUS_DIR}')


In [ ]:
# Convert + save 627 test queries with gold labels
for record in tqdm(queries['test_queries'], desc='Exporting queries'):
    doc = ilpcsr_to_caserag(record)
    # Attach gold labels for evaluation
    doc['relevant_precedent_ids'] = record['relevant_precedent_ids']
    doc['relevant_statute_ids']   = record['relevant_statute_ids']
    path = os.path.join(QUERIES_DIR, f"{doc['caseID']}.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f'Saved {len(os.listdir(QUERIES_DIR))} query files to {QUERIES_DIR}')

# Peek at one converted query
sample_path = os.path.join(QUERIES_DIR, os.listdir(QUERIES_DIR)[0])
with open(sample_path) as f:
    sample_out = json.load(f)
print(list(sample_out.keys()))
print('Gold precedents:', sample_out['relevant_precedent_ids'])
